In [ ]:
import os
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, AIMessageChunk, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# ========== CONFIGURAÇÃO DOS LLMs ==========
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_tradutor = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ========== FERRAMENTAS ==========
@tool
def calculadora(expressao: str) -> str:
    """Ferramenta para realizar cálculos matemáticos."""
    try:
        import math
        resultado = eval(expressao, {"__builtins__": {}}, {
            "sqrt": math.sqrt,
            "pow": math.pow,
            "abs": abs,
            "round": round,
            "math": math
        })
        return f"Resultado: {resultado}"
    except Exception as e:
        return f"Erro no cálculo: {str(e)}"

@tool
def tradutor(texto: str, idioma_destino: str = "inglês") -> str:
    """Ferramenta para traduzir textos usando LLM interno."""
    try:
        prompt = f"Traduza o seguinte texto para {idioma_destino}. Retorne APENAS a tradução:\n\n{texto}"
        response = llm_tradutor.invoke(prompt)
        return f"Tradução: {response.content.strip()}"
    except Exception as e:
        return f"Erro na tradução: {str(e)}"

tools = [calculadora, tradutor]

# ========== PROMPT DO AGENTE ==========
AGENT_PROMPT = """
Você é um assistente especializado que atua de forma explicativa e didática, com ferramentas de cálculo e tradução.

REGRA OBRIGATÓRIA DE FORMATAÇÃO:
Toda resposta DEVE seguir este formato exato:

[Explicação detalhada do raciocínio, contexto e processo]

[Linha em branco]

[Resultado final claro e direto]

Exemplos:

Pergunta: "Quanto é 1 + 1 * 2?"
Resposta correta:
Esse cálculo contém uma soma e uma multiplicação. Na matemática, seguimos a ordem de precedência das operações, onde a multiplicação vem antes da adição. Primeiro calculamos 1 * 2 = 2, e depois somamos 1 ao resultado.

O resultado de 1 + 1 * 2 é igual a 3.

---

Pergunta: "Traduza 'Bom dia' para francês"
Resposta correta:
A expressão "Bom dia" é uma saudação matinal comum em português. Em francês, a saudação equivalente combina "bon" (bom) com "jour" (dia).

A tradução de "Bom dia" para francês é "Bonjour".

---

SEMPRE siga esse padrão: explicação primeiro, linha em branco, depois resultado final.
"""

# ========== CRIAÇÃO DO AGENTE ==========
agent = create_react_agent(llm, tools)


# ========== FUNÇÃO 1: RUN SIMPLES ==========
def run_agent(user_input: str, max_iterations: int = 15) -> str:
    """
    Executa o agente e retorna a resposta final completa.
    
    Args:
        user_input: Pergunta do usuário
        max_iterations: Número máximo de iterações para evitar loops
    
    Returns:
        String com a resposta completa
    """
    messages = [
        SystemMessage(content=AGENT_PROMPT),
        HumanMessage(content=user_input)
    ]
    
    result = agent.invoke(
        {"messages": messages},
        {"recursion_limit": max_iterations}
    )
    
    return result["messages"][-1].content


# ========== FUNÇÃO 2: RUN COM STREAMING ==========
def run_agent_stream(user_input: str, max_iterations: int = 15) -> str:
    """
    Executa o agente com streaming palavra por palavra.
    Mostra o texto sendo gerado em tempo real.
    
    Args:
        user_input: Pergunta do usuário
        max_iterations: Número máximo de iterações para evitar loops
    
    Returns:
        String com a resposta completa
    """
    messages = [
        SystemMessage(content=AGENT_PROMPT),
        HumanMessage(content=user_input)
    ]
    
    resposta_final = ""
    
    try:
        # stream_mode="messages" captura cada token individual
        for event in agent.stream(
            {"messages": messages},
            stream_mode="messages",
            {"recursion_limit": max_iterations}
        ):
            # event é uma tupla (mensagem, metadata)
            msg, metadata = event
            
            # Detecta uso de ferramentas
            if isinstance(msg, AIMessage) and hasattr(msg, "tool_calls") and msg.tool_calls:
                for tool_call in msg.tool_calls:
                    print(f"\n[🔧 {tool_call['name']}] ", end="", flush=True)
            
            # Streaming de conteúdo (token por token)
            elif isinstance(msg, AIMessageChunk):
                if hasattr(msg, "content") and msg.content:
                    print(msg.content, end="", flush=True)
                    resposta_final += msg.content
            
            # Captura mensagem AI completa (fallback)
            elif isinstance(msg, AIMessage) and hasattr(msg, "content") and msg.content:
                # Evita duplicação
                if msg.content not in resposta_final:
                    print(msg.content, end="", flush=True)
                    resposta_final = msg.content
        
        print()  # Nova linha no final
        return resposta_final
        
    except Exception as e:
        print(f"\n❌ Erro no streaming: {e}")
        # Fallback para execução normal
        print("⚠️ Executando sem streaming...")
        return run_agent(user_input, max_iterations)


# ========== EXEMPLOS DE USO ==========
if __name__ == "__main__":
    print("="*60)
    print("TESTE 1: Run simples (sem streaming)")
    print("="*60)
    resposta = run_agent("Quanto é 2 + 2?")
    print(resposta)
    
    print("\n" + "="*60)
    print("TESTE 2: Run com streaming (palavra por palavra)")
    print("="*60)
    run_agent_stream("Traduza 'Olá mundo' para inglês")
    
    print("\n" + "="*60)
    print("TESTE 3: Cálculo com streaming")
    print("="*60)
    run_agent_stream("Quanto é 15 * 8?")